# Recommender System - Collaborative Filtering
This notebook documents a practical user-based Collaborative Filtering pipeline using the MovieLens 100K dataset.
It complements a recommendation systems presentation and focuses on implementation decisions and outputs.



## Objective
Build a user-item matrix from ratings.
Compute user-to-user similarity.
Generate top-N recommendations for a target user.


In [17]:
from src import (load_ratings, load_movies, build_user_item_matrix,
    fill_missing_values, compute_sparsity, compute_user_similarity,
    recommend_user_based)


## Data Loading
MovieLens 100K provides user IDs, item IDs, and explicit ratings.
Movies metadata supplies titles so recommendations are readable.
The dataset is structured for quick joins between ratings and items.


In [18]:
ratings_df = load_ratings("datasets/ratings.csv")
movies_df = load_movies("datasets/movies.csv")

In [ ]:
from IPython.display import display
display(ratings_df.head())
display(movies_df.head())



## Data Exploration
Exploration validates data integrity and shapes before modeling.
We check the number of users and items, then inspect rating distribution to understand bias in feedback frequency.


In [20]:
ratings_df.head()
ratings_df.shape

(100836, 4)


## Sparsity Analysis
Sparsity is the fraction of missing user-item ratings in the matrix.
Collaborative Filtering relies on user interactions, but in real datasets most entries are missing, which makes similarity harder to compute.
High sparsity reduces overlap and weakens similarity signals.


In [21]:
sparsity = compute_sparsity(ratings_df)


## User-Item Matrix
Rows represent users and columns represent items.
Each cell holds a rating, with missing values treated as "not rated."
This matrix is the core structure for computing similarities and generating recommendations.


In [22]:
user_item_matrix = build_user_item_matrix(ratings_df)

user_item_filled = fill_missing_values(user_item_matrix)


## Similarity Computation
User similarity is computed by comparing rating vectors.
Cosine similarity measures the angle between vectors, focusing on preference alignment rather than rating scale.
Higher similarity indicates more aligned taste.


In [23]:
user_similarity = compute_user_similarity(user_item_filled)


## Recommendation Logic
Select a target user.
Find similar users based on cosine similarity.
Aggregate similar users' ratings with a weighted average to score unseen items.
Rank scores to produce top-N recommendations.


In [ ]:
recommendations = recommend_user_based(
    user_id=1,
    user_item_matrix=user_item_filled,
    similarity_matrix=user_similarity,
    movies_df=movies_df
)

recommendations.head()

## Example
This section shows a single user's top-rated movies and the top recommendations generated by user-based collaborative filtering.
It's meant as a quick sanity check that the pipeline is producing readable, sensible outputs before tuning hyperparameters.


In [ ]:
target_user = 1

# Show movies the user has already rated (top 10 by rating)
user_ratings = ratings_df[ratings_df["user_id"] == target_user]
user_ratings = user_ratings.merge(
    movies_df[["item_id", "title"]],
    on="item_id",
    how="left"
)
user_ratings = user_ratings.sort_values("rating", ascending=False).head(10)

from IPython.display import display
print(f"Top ratings by user {target_user}")
display(user_ratings[["title", "rating"]])

# Show recommendations
print(f"Top recommendations for user {target_user}")
display(recommendations[["title", "score"]].head(10))


## Limitation
User-based CF struggles with cold-start users and items, and it can be sensitive to sparsity.
Similarity scores may be noisy when users share very few co-rated items, which can reduce recommendation quality.
